# zReg — Debug / Development Notebook

This notebook is a development and debugging companion for zReg. It loads two
`.tracklets` trajectories and one Shah CSV dataset, and provides a visualisation
helper for inspecting point clouds interactively. It is intended for developers
exploring the codebase or diagnosing registration issues, not as a polished tutorial.

## MPI note

`mpi4py` is an **optional** dependency. If it is not installed, zreg falls back to
single-process mode automatically — no environment variable or workaround is needed.
If you do have `mpi4py` installed and want to suppress MPI initialisation, pass
`mpi_distribute=False` to the functions that support it (e.g. `DTW`, `pairwise_distance_matrix`).

In [ ]:
DATA_PATH     = "# Replace with the path to your first .tracklets file"
DATA_PATH_2   = "# Replace with the path to your second .tracklets file"
DATA_PATH_CSV = "# Replace with the path to your Shah CSV file"

import zreg
import copy
import time
import torch

%load_ext autoreload
%autoreload 2

## Load trajectory data

Two `.tracklets` datasets and one Shah CSV are loaded here. `pcs` and `pcs2` are
time-series from two different embryo recordings; `pcs3` is loaded from a CSV file
in the Shah et al. format. If you only have some of these files, comment out the
lines you do not need — the remaining cells can be adapted accordingly.

In [ ]:
pcs, _alltracklets = zreg.dataset.load_data_from_tracklets(
    filepath=DATA_PATH,
    device="cpu",
)
pcs2, _alltracklets = zreg.dataset.load_data_from_tracklets(
    filepath=DATA_PATH_2,
    device="cpu",
)
pcs3 = zreg.dataset.load_shah_from_csv(
    filepath=DATA_PATH_CSV,
    device="cpu",
)

## Visualisation helper

`draw_registration_result` accepts `zRegPointCloud` dicts and renders them in a
matplotlib 3-D scatter. The `target` argument is optional: if omitted, only the
source is displayed — useful for inspecting a single frame or a transformed result
in isolation.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt


def _to_numpy(pc):
    pos = pc["pos"] if isinstance(pc, dict) else pc
    return pos.detach().cpu().numpy() if hasattr(pos, "detach") else pos


def draw_registration_result(source, target=None):
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")
    src = _to_numpy(source)
    ax.scatter(src[:, 0], src[:, 1], src[:, 2],
               c="#FFB300", s=1.5, alpha=0.8, linewidths=0, label="Source")
    if target is not None:
        tgt = _to_numpy(target)
        ax.scatter(tgt[:, 0], tgt[:, 1], tgt[:, 2],
                   c="#1976D2", s=1.5, alpha=0.8, linewidths=0, label="Target")
        ax.legend(markerscale=8)
    ax.set_xlabel("x", fontsize=8)
    ax.set_ylabel("y", fontsize=8)
    ax.set_zlabel("z", fontsize=8)
    plt.tight_layout()
    plt.show()